In [1]:
import os
#import h5py
import scipy.io
import re
import numpy as np
from pyemittance.emittance_calc import EmitCalc

In [2]:
scans_dir = f"matlab_scans/"
regex = r"^Emittance-scan-OTRS_IN20_541-2022-[0-9]{2}-[0-9]{2}-[0-9]{6}\.mat$"


filenames=[]
for file in os.scandir(scans_dir):
    if re.match(regex, file.name):
        filenames.append(file.name)
print('Total ',len(filenames),'files')

Total  9 files


In [3]:
fname = filenames[0]

In [5]:
fname = 'Emittance-scan-OTRS_IN20_541-2022-01-25-055155.mat'

f = scipy.io.loadmat(scans_dir + fname)
data = f['data']

In [6]:
mat = scipy.io.loadmat(scans_dir + filenames[0])
beam_entry = mat['data'][0]['beam'][0][0, 0]
print(beam_entry.dtype.names)

('profx', 'xStat', 'xStatStd', 'profy', 'yStat', 'yStatStd', 'profu', 'uStat', 'uStatStd', 'method', 'stats', 'statsStd')


In [8]:
import json
def load_configs(dir_name="LCLS_OTR2"):
    all_data = {}
    for json_name in json_namelist:
        fname = os.path.join(dir_name, json_name + ".json")
        print(fname)
        if os.path.exists(fname):
            data = json.load(open(fname))
        elif json_name == 'savepaths':
            # Special case
            data = {"summaries": None,
                    "images": None,
                     }
        else:
            raise FileNotFoundError(
                f"*** File '{json_name}.json' does not exist,"
                f" please create appropriate json file for configuration. *** \n"
                f"*** Or alternatively, initialize EmitCalc with dict directly. ***"
            )
        
        # Add to all data
        all_data[json_name] = data

    # dict of all configs
    return all_data

In [9]:
json_namelist = [
    "beamline_info",
    "img_proc",
    "meas_pv_info",
    "savepaths", # optional
]
config = load_configs()

LCLS_OTR2/beamline_info.json
LCLS_OTR2/img_proc.json
LCLS_OTR2/meas_pv_info.json
LCLS_OTR2/savepaths.json


In [19]:
mat_data["quadName"]

array([array(['QUAD:IN20:425'], dtype='<U13')], dtype=object)

In [29]:
import scipy.io

for fname in filenames:
    print(fname)
    mat = scipy.io.loadmat(scans_dir + fname)
    mat_data = mat['data'][0]  # the top-level struct

    num_steps = mat_data['beam'][0].shape[0]
    quadvals = mat_data['quadVal'][0].flatten()

    xrms, yrms, xrms_err, yrms_err = [], [], [], []
    for j in range(num_steps):
        beam_j = mat_data['beam'][0][j, 0]   # method 0
        xrms.append(beam_j['stats'][0, 2])
        yrms.append(beam_j['stats'][0, 3])
        xrms_err.append(beam_j['statsStd'][0, 2])
        yrms_err.append(beam_j['statsStd'][0, 3])

    xrms, yrms = np.array(xrms)*1e-6, np.array(yrms)*1e-6
    xrms_err, yrms_err = np.array(xrms_err)*1e-6, np.array(yrms_err)*1e-6

    try:
        twiss = mat_data['twiss'][0].flatten()
    except (KeyError, IndexError):
        print(f"Error in file {fname}.\nSkipping.\n")
        continue
    twissstd = mat_data['twissstd'][0].flatten()
    emitx = twiss[8]
    emitx_err = twissstd[8]
    bmagx = twiss[11]
    bmagx_err = twissstd[11]
    emity = twiss[12]
    emity_err = twissstd[12]
    bmagy = twiss[15]
    bmagy_err = twissstd[15]

    twiss0 = mat_data['twiss0'][0].flatten()

    # pass this info to the Emittance Calc class to start a measurement
    ef = EmitCalc({'x': quadvals, 'y': quadvals},
                  {'x': xrms, 'y': yrms},
                  {'x': xrms_err, 'y': yrms_err},
                  config_dict = config
                  )

    '''' MAKE ANY ADJUSTMENTS TO LCLS2_OTR3 NEEDED AS RUN ORIGINALLY '''

    ef.plot = False
    ef.calc_bmag = True

    # get normalized transverse emittance, the out_dict is the results returned in a dictionary
    out_dict = ef.get_emit()

    if out_dict["error_x"] or out_dict["error_y"]:
        print(f"Error in file {fname}.\nSkipping.\n")
        continue

    print("\033[1;3mpyemittance\033[0m")
    print(f"norm_emitx: {out_dict['norm_emit_x']/1e-6:.2f} +/- {out_dict['norm_emit_x_err']/1e-6:.2f} um")
    print(f"norm_emity: {out_dict['norm_emit_y']/1e-6:.2f} +/- {out_dict['norm_emit_y_err']/1e-6:.2f} um")
    print(f"bmagx: {out_dict['screen_bmagx']:.2f} +/- {out_dict['screen_bmagx_err']:.2f}")
    print(f"bmagy: {out_dict['screen_bmagy']:.2f} +/- {out_dict['screen_bmagy_err']:.2f}")

    print("\033[1;3mmatlab\033[0m")
    print(f"norm_emitx: {emitx/1e-6:.2f} +/- {emitx_err/1e-6:.2f} um")
    print(f"norm_emity: {emity/1e-6:.2f} +/- {emity_err/1e-6:.2f} um")
    print(f"bmagx: {bmagx:.2f} +/- {bmagx_err:.2f}")
    print(f"bmagy: {bmagy:.2f} +/- {bmagy_err:.2f}")

    print("\033[1;3mconfigs\033[0m")
    print("PyEmit Twiss0: ", ["%E" % e for e in ef.config_dict['beamline_info']['Twiss0']])
    print("Matlab Twiss0: ", ["%E" % e for e in twiss0])

    print("PyEmit r-matx: ",ef.config_dict['beamline_info']['rMatx'])
    print("PyEmit r-maty: ",ef.config_dict['beamline_info']['rMaty'])
    print("\n")

    #break


Emittance can't be computed. Returning error
Emittance can't be computed. Returning error
Emittance can't be computed. Returning error
Emittance can't be computed. Returning error
Emittance can't be computed. Returning error


Emittance-scan-OTRS_IN20_541-2022-01-25-055155.mat
pyemittance
norm_emitx: 0.80 +/- 0.04 um
norm_emity: 0.60 +/- 0.01 um
bmagx: 2.01 +/- 0.10
bmagy: 1.01 +/- 0.01
matlab
norm_emitx: 0.45 +/- 0.02 um
norm_emity: 0.55 +/- 0.02 um
bmagx: 0.00 +/- 0.00
bmagy: 1.18 +/- 0.11
configs
PyEmit Twiss0:  ['1.000000E-06', '1.000000E-06', '1.113081E+00', '1.113022E+00', '-6.894036E-02', '-7.029490E-02']
Matlab Twiss0:  ['1.000000E-06', '1.000000E-06', '4.161577E+00', '4.161577E+00', '1.663280E+00', '1.663280E+00']
PyEmit r-matx:  [1, 2.26, 0, 1]
PyEmit r-maty:  [1, 2.26, 0, 1]


Emittance-scan-OTRS_IN20_541-2022-01-23-012229.mat
pyemittance
norm_emitx: 0.99 +/- 0.32 um
norm_emity: 0.73 +/- 0.01 um
bmagx: 1.79 +/- 0.58
bmagy: 1.08 +/- 0.01
matlab
norm_emitx: 0.60 +/- 0.06 um
norm_emity: 0.61 +/- 0.01 um
bmagx: 0.00 +/- 0.00
bmagy: 4.29 +/- 2.65
configs
PyEmit Twiss0:  ['1.000000E-06', '1.000000E-06', '1.113081E+00', '1.113022E+00', '-6.894036E-02', '-7.029490E-02']
Matlab Twiss0:  ['1.000000E-06', '1

pyemittance
norm_emitx: 0.75 +/- 0.04 um
norm_emity: 0.56 +/- 0.01 um
bmagx: 2.11 +/- 0.10
bmagy: 1.01 +/- 0.01
matlab
norm_emitx: 0.45 +/- 0.02 um
norm_emity: 0.55 +/- 0.02 um
bmagx: 0.00 +/- 0.00
bmagy: 1.18 +/- 0.11
configs
PyEmit Twiss0:  ['1.000000E-06', '1.000000E-06', '1.113081E+00', '1.113022E+00', '-6.894036E-02', '-7.029490E-02']
Matlab Twiss0:  ['1.000000E-06', '1.000000E-06', '4.161577E+00', '4.161577E+00', '1.663280E+00', '1.663280E+00']
PyEmit r-matx:  [1, 2.26, 0, 1]
PyEmit r-maty:  [1, 2.26, 0, 1]


pyemittance
norm_emitx: 0.90 +/- 0.30 um
norm_emity: 0.68 +/- 0.01 um
bmagx: 1.90 +/- 0.63
bmagy: 1.09 +/- 0.01
matlab
norm_emitx: 0.45 +/- 0.02 um
norm_emity: 0.55 +/- 0.02 um
bmagx: 0.00 +/- 0.00
bmagy: 1.18 +/- 0.11
configs
PyEmit Twiss0:  ['1.000000E-06', '1.000000E-06', '1.113081E+00', '1.113022E+00', '-6.894036E-02', '-7.029490E-02']
Matlab Twiss0:  ['1.000000E-06', '1.000000E-06', '4.161577E+00', '4.161577E+00', '1.663280E+00', '1.663280E+00']
PyEmit r-matx:  [1, 2.26, 0, 1]
PyEmit r-maty:  [1, 2.26, 0, 1]


pyemittance
norm_emitx: 0.79 +/- 0.01 um
norm_emity: 0.43 +/- 0.01 um
bmagx: 1.12 +/- 0.01
bmagy: 1.18 +/- 0.04
matlab
norm_emitx: 0.45 +/- 0.02 um
norm_emity: 0.55 +/- 0.02 um
bmagx: 0.00 +/- 0.00
bmagy: 1.18 +/- 0.11
configs
PyEmit Twiss0:  ['1.000000E-06', '1.000000E-06', '1.113081E+00', '1.113022E+00', '-6.894036E-02', '-7.029490E-02']
Matlab Twiss0:  ['1.000000E-06', '1.000000E-06', '4.161577E+00', '4.161577E+00', '1.663280E+00', '1.663280E+00']
PyEmit r-matx:  [1, 2.26, 0, 1]
PyEmit r-maty:  [1, 2.26, 0, 1]


pyemittance
norm_emitx: 0.75 +/- 0.02 um
norm_emity: 0.48 +/- 0.02 um
bmagx: 1.24 +/- 0.03
bmagy: 1.18 +/- 0.04
matlab
norm_emitx: 0.45 +/- 0.02 um
norm_emity: 0.55 +/- 0.02 um
bmagx: 0.00 +/- 0.00
bmagy: 1.18 +/- 0.11
configs
PyEmit Twiss0:  ['1.000000E-06', '1.000000E-06', '1.113081E+00', '1.113022E+00', '-6.894036E-02', '-7.029490E-02']
Matlab Twiss0:  ['1.000000E-06', '1.000000E-06', '4.161577E+00', '4.161577E+00', '1.663280E+00', '1.663280E+00']
PyEmit r-matx:  [1, 2.26, 0, 1]
PyEmit r-maty:  [1, 2.26, 0, 1]


Error in file Emittance-scan-OTRS_IN20_541-2022-01-23-011437.mat.
Skipping.

Error in file Emittance-scan-OTRS_IN20_541-2022-01-23-021526.mat.
Skipping.

Error in file Emittance-scan-OTRS_IN20_541-2022-01-23-021311.mat.
Skipping.

pyemittance
norm_emitx: 0.83 +/- 0.01 um
norm_emity: 0.47 +/- 0.01 um
bmagx: 1.14 +/- 0.01
bmagy: 1.09 +/- 0.03
matlab
norm_emitx: 0.45 +/- 0.02 um
norm_emity: 0.55 +/- 0.02 um
bmagx: 0.00 +/- 0.00
bmagy: 1.18 +/- 0.11
configs
PyEmit Twiss0:  ['1.000000E-06', '1.000000E-06', '1.113081E+00', '1.113022E+00', '-6.894036E-02', '-7.029490E-02']
Matlab Twiss0:  ['1.000000E-06', '1.000000E-06', '4.161577E+00', '4.161577E+00', '1.663280E+00', '1.663280E+00']
PyEmit r-matx:  [1, 2.26, 0, 1]
PyEmit r-maty:  [1, 2.26, 0, 1]


Error in file Emittance-scan-OTRS_IN20_541-2022-01-23-014733.mat.
Skipping.

In [11]:
# Are the 7 entries repeated measurements to average?
for j in range(7):
    print(f"beam[0,{j}]['xStat']:", data['beam'][0, j]['xStat'])

beam[0,0]['xStat']: [[ 2.47157330e+05 -9.20223644e+01  5.43493973e+02  0.00000000e+00
   0.00000000e+00]]
beam[0,1]['xStat']: [[ 2.46312236e+05 -1.21575476e+02  5.41811180e+02 -1.71419872e-01
   2.10017676e-02]]
beam[0,2]['xStat']: [[ 2.58850616e+05 -8.71153324e+01  5.93571197e+02  0.00000000e+00
   4.06303727e-01]]
beam[0,3]['xStat']: [[ 2.68215000e+05 -9.75242746e+01  6.40653037e+02  1.74304224e-02
   8.35338865e-01]]
beam[0,4]['xStat']: [[ 2.64212000e+05 -1.02636577e+02  6.00604834e+02  7.22060077e-03
   3.01672230e-01]]
beam[0,5]['xStat']: [[ 2.55123000e+05 -1.00522006e+02  5.34826306e+02  3.98525399e-05
  -3.88182526e-01]]
beam[0,6]['xStat']: [[ 3.02694000e+05 -8.24603395e+01  9.31531440e+02  6.08722690e-02
   7.85018389e-01]]


In [8]:
mat = scipy.io.loadmat(scans_dir + filenames[0])
beam = mat['data'][0,0]['beam'][0,0]
xstat = beam['xStat']
print("xStat shape:", xstat.shape)
print("xStat dtype:", xstat.dtype)

# look at first element
elem = xstat.flat[0]
print("first elem:", elem)
print("first elem shape:", elem.shape)

xStat shape: (1, 5)
xStat dtype: float64
first elem: 247157.32994101898
first elem shape: ()


# LUME

In [13]:
from lcls_cu_inj_model import load_model
model = load_model()
print(model.evaluate({"QUAD:IN20:425:BCTRL": -1}))

/Users/smiskov/miniconda3/envs/agentic-ws-311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'OTRS:IN20:571:XRMS': tensor([[281.6813]], dtype=torch.float64), 'OTRS:IN20:571:YRMS': tensor([[119.9943]], dtype=torch.float64), 'sigma_z': tensor([[0.0005]], dtype=torch.float64), 'norm_emit_x': tensor([[5.6149e-07]], dtype=torch.float64), 'norm_emit_y': tensor([[5.6160e-07]], dtype=torch.float64)}


In [48]:
with open("pv_data.json", "r") as f:
    loaded_data = json.load(f)

In [49]:
for fname in filenames:
    print(fname)
    # calculate r_dist
    # update quad scan value
    input_dict = loaded_data[fname].copy()

    input_dict["CAMR:IN20:186:R_DIST"] = (input_dict["CAMR:IN20:186:XRMS"]**2 +input_dict["CAMR:IN20:186:YRMS"]**2)**0.5 
    del input_dict["CAMR:IN20:186:XRMS"]
    del input_dict["CAMR:IN20:186:YRMS"]

    # do scan
    mat = scipy.io.loadmat(scans_dir + fname)
    mat_data = mat['data'][0]
    quadvals = mat_data['quadVal'][0].flatten()
    for q in quadvals:
        print("step value: ", q)
        input_dict["QUAD:IN20:425:BCTRL"] = float(q)
        results = model.evaluate(input_dict)
        print(results["norm_emit_x"].item()/1e-6, results["norm_emit_y"].item()/1e-6)
        
    # adjust so that it's scanning for xrms, yrms at each step and calculates the emittance like the code above
    # compare to the NN's norm_emit values that are predicted at each step
    
    

Emittance-scan-OTRS_IN20_541-2022-01-25-055155.mat
step value:  -7.0
1.3315025732073924 1.0791560532768292
step value:  -4.333333333333334
1.2877670984810348 1.0409537813467193
step value:  -1.666666666666667
1.1916488732060806 1.0201246787554379
step value:  1.0
1.159336673768268 1.0195221838816633
Emittance-scan-OTRS_IN20_541-2022-01-23-012229.mat
step value:  -5.0
1.3574741900068348 1.07726035435656
step value:  -3.666666666666667
1.3163462265693613 1.057345801921486
step value:  -2.3333333333333335
1.2653572623329081 1.0551932170550509
step value:  -1.0
1.2323498215155684 1.0479353457032814
Emittance-scan-OTRS_IN20_541-2022-01-26-030852.mat
step value:  -3.0
1.2312396050344998 1.016841275352672
step value:  -1.6666666666666667
1.1838032760024315 1.0136255562385688
step value:  -0.3333333333333335
1.1532252675029104 0.9929548940456084
step value:  1.0
1.1497586963344375 1.0102886887002114
Emittance-scan-OTRS_IN20_541-2022-01-23-014920.mat
step value:  -3.0
1.2870591439119519 1.05209

In [23]:
results

{'OTRS:IN20:571:XRMS': tensor([[3849.7341]], dtype=torch.float64),
 'OTRS:IN20:571:YRMS': tensor([[1525.6209]], dtype=torch.float64),
 'sigma_z': tensor([[0.0005]], dtype=torch.float64),
 'norm_emit_x': tensor([[1.4262e-06]], dtype=torch.float64),
 'norm_emit_y': tensor([[1.2335e-06]], dtype=torch.float64)}